# UTM — KPIs selecionados para dados sintéticos

Modelo focado nos dois KPIs de UTM priorizados para o artigo 2: **UTM-CAP-01** (Taxa de Acomodação da Demanda) e **UTM-PRED-01** (Taxa de Previsibilidade do Horário das Operações). Ancorado no ConOps UTM v2.0 da FAA.

Execute as células **em ordem**: primeiro o núcleo comum, depois o modelo focado. Fórmulas, τ e Δ são **propostas** para o artigo 2.

## Núcleo comum

Idêntico ao `KPI_Common.ipynb`.

In [ ]:
package KPI_Taxonomy {
    doc /* Enumerations and metadata used to classify KPAs, performance objectives and KPI candidates (article 1: applicability, analytical status, measurement maturity, taxonomic fit). */
    private import ScalarValues::*;

    enum def Domain { enum UTM; enum AAM; }
    enum def Applicability { enum Applicable; enum ConditionallyApplicable; enum NotApplicable; }
    enum def RetentionStatus { enum ProvisionallyRetained; enum ConditionallyRetained; }
    enum def Operationalization {
        doc /* Measurement maturity of the candidate, kept distinct from retention status (article 1). */
        enum ConceptualFormulaInArticle1;
        enum ProposedInThisStudy;
        enum Pending;
    }
    enum def TaxonomicFit { enum Direct; enum Provisional; enum Gap; }

    metadata def KPAApplicability {
        attribute domain : Domain;
        attribute applicability : Applicability;
    }
    metadata def CandidateKPI {
        attribute code : String;
        attribute domain : Domain;
        attribute measurementConstruct : String;
        attribute status : RetentionStatus;
        attribute maturity : Operationalization;
    }
    metadata def DomainExtension {
        doc /* Construct that extends the reference KPA taxonomy for the domain. */
        attribute fit : TaxonomicFit;
    }
    metadata def SupportingDiagnosticMetric {
        doc /* Explains why a system-level KPI behaves as observed; not itself a system-level KPI. */
    }

    enum def SourceClass {
        doc /* Where a KPI input exists according to the FAA concepts of operations (experiment E1).
         * NetworkShared: exchanged across the USS Network or the federated service network during operations.
         * ProviderArchived: retained by USSs, PSUs or operators and available on authorized request.
         * InternallyHeld: exists within one operator or provider, with no sharing or archiving requirement.
         * NotAddressed: no corresponding data element in the concept of operations. */
        enum NetworkShared;
        enum ProviderArchived;
        enum InternallyHeld;
        enum NotAddressed;
    }
    enum def InputRole {
        doc /* Principal: the quantity the KPI counts or measures; Supporting: any other input. */
        enum Principal;
        enum Supporting;
    }
    attribute def KPIInput {
        doc /* A quantity consumed by a KPI candidate, described in its doc and classified by @DataSource. */
    }
    metadata def DataSource {
        doc /* Source class of a data element in the FAA concepts of operations, with the section and printed page. */
        attribute sourceClass : SourceClass;
        attribute conopsRef : String;
    }
    metadata def KPIInputRole {
        doc /* Role of an input in a KPI; on a KPIInput attribute, or on a dependency from a KPI to a data-item attribute. */
        attribute role : InputRole;
    }
}

In [ ]:
package ICAO_KPAs {
    doc /* The eleven ICAO ATM reference KPAs (ICAO Doc 9854 and Doc 9883), used as the common performance reference structure. Performance Intents: article 1, Table 1. */

    concern def AccessAndEquity { doc /* ICAO reference KPA: Access and Equity. */ }
    concern def Capacity { doc /* ICAO reference KPA: Capacity. */ }
    concern def CostEffectiveness { doc /* ICAO reference KPA: Cost Effectiveness. */ }
    concern def Efficiency { doc /* ICAO reference KPA: Efficiency. */ }
    concern def Environment { doc /* ICAO reference KPA: Environment. */ }
    concern def Flexibility { doc /* ICAO reference KPA: Flexibility. */ }
    concern def GlobalInteroperability { doc /* ICAO reference KPA: Global Interoperability. */ }
    concern def Participation { doc /* ICAO reference KPA: ATM Community Participation. */ }
    concern def Predictability { doc /* ICAO reference KPA: Predictability. */ }
    concern def Safety { doc /* ICAO reference KPA: Safety. */ }
    concern def Security { doc /* ICAO reference KPA: Security. */ }
}

In [ ]:
package System_Boundaries {
    doc /* Selected system boundaries (article 1, Sec. 2.4). */

    part def UTMSystem {
        doc /* Based on the FAA UTM ConOps v2.0: traffic-management services, operational coordination,
         * information exchange, service interactions and constraint management that support
         * unmanned operations. Intrinsic aircraft or propulsion characteristics are excluded. */
    }
    part def AAMSystem {
        doc /* Based on the FAA UAM ConOps v2.0 and AAM Implementation Plan: an integrated mobility
         * system of systems with aircraft operations, traffic-management interactions, vertiports,
         * supporting infrastructure, energy, digital services, ground access, users and affected
         * communities. UAM is treated as the urban subset of AAM. */
    }
}

In [ ]:
package KPI_Patterns {
    doc /* Common calculation patterns shared by UTM and AAM candidates. PROPOSED for article 2; not defined in article 1. */
    private import ScalarValues::*;

    abstract calc def AccommodationRate {
        doc /* Accommodated demand over valid demand in a period and volume, in percent. */
        in accommodated : Natural;
        in validDemand : Natural;
        return rate : Real = accommodated / validDemand * 100;
    }
    abstract calc def TemporalConformanceRate {
        doc /* Completed operations whose actual time deviates from the planned time by no more
         * than the tolerance, over all completed operations, in percent. */
        in withinTolerance : Natural;
        in completed : Natural;
        in toleranceSeconds : Real;
        return rate : Real = withinTolerance / completed * 100;
    }
}

## UTM_Selected

O modelo focado traz a arquitetura mínima do ConOps, os itens de dados que alimentam os KPIs, os parâmetros em aberto, os dois objetivos de desempenho, os dois KPIs, as métricas diagnósticas, a rastreabilidade até os atributos de dados e o `analysis def` de uma rodada sintética.

In [ ]:
package UTM_Selected {
    doc /* Focused model for the two UTM KPIs prioritized for synthetic data in article 2:
     * UTM-CAP-01 (Demand Accommodation Rate) and UTM-PRED-01 (Operation-Time Predictability Rate).
     * Anchored in the FAA UTM ConOps v2.0. Only the elements needed to compute the two KPIs are
     * modeled (minimum-sufficiency criterion). Formulas and parameters are PROPOSED for article 2. */
    private import ScalarValues::*;
    private import ICAO_KPAs::*;
    private import KPI_Taxonomy::*;
    private import KPI_Patterns::*;
    private import System_Boundaries::*;

    // ---------- Architecture (FAA UTM ConOps v2.0, Sec. 2.3) ----------

    part def Operator {
        doc /* Operator: plans the operation and shares operation intent, via its USS
         * (UTM ConOps Sec. 2.3.1, p. 9; Sec. 2.4.4, pp. 17-18). */
    }
    part def USS {
        doc /* UAS Service Supplier: communications bridge between federated actors, supports strategic
         * de-confliction and conformance monitoring, and archives operations data for analytics,
         * regulatory and accountability purposes (UTM ConOps Sec. 2.3.2, p. 11; App. C, pp. 63-64). */
        ref item receivedIntents : OperationIntent[0..*];
        ref item archivedRecords : OperationRecord[0..*];
    }
    part def USSNetwork {
        doc /* USS Network: USSs connected to each other, sharing operation intent and airspace
         * constraint information (UTM ConOps Sec. 2.3.2, p. 11). */
    }
    part def FIMS {
        doc /* Flight Information Management System: interface for data exchange between FAA systems and
         * UTM participants; also the access point for post-hoc archived data (UTM ConOps Sec. 2.3.2, p. 12). */
    }
    part def UTMEnvironment :> UTMSystem {
        doc /* Instance of the selected UTM boundary used for the synthetic study. */
        part operators : Operator[1..*];
        part ussProviders : USS[2..*];
        part network : USSNetwork[1];
        part fims : FIMS[1];
    }

    // ---------- Data items consumed by the KPIs ----------

    enum def IntentOutcome {
        doc /* Planning outcome used by the synthetic study. Accepted and Withdrawn follow the ConOps
         * (Sec. 3.2, p. 39; Sec. 2.7.3, p. 33). The Rejected outcomes are not ConOps states: a plan that
         * cannot be de-conflicted is simply not shared, so these outcomes are internally held. */
        enum Accepted;
        enum RejectedConflict;
        enum RejectedConstraint;
        enum RejectedInvalid;
        enum Withdrawn;
    }
    item def OperationIntent {
        doc /* Operation intent: Operation Volume segments with entry and exit times (UTM ConOps Sec. 2.7.1,
         * p. 23), shared with the USS Network once planning is complete; the operation is then in the
         * Accepted state (Sec. 3.2, p. 39). requestedStart comes from the initial planning information
         * the operator gives its USS (p. 37) and is internally held: the ConOps defines no sharing or
         * archiving of it. Times in seconds from the start of the measurement period. */
        attribute intentId : String;
        attribute isValid : Boolean { @DataSource { sourceClass = SourceClass::InternallyHeld; conopsRef = "UTM §2.4.4, p. 17; §3.2, p. 37"; } }
        attribute outcome : IntentOutcome { @DataSource { sourceClass = SourceClass::NetworkShared; conopsRef = "UTM §2.4.4, p. 18; §3.2, p. 39"; } }
        attribute requestedStart : Real { @DataSource { sourceClass = SourceClass::InternallyHeld; conopsRef = "UTM §3.2, p. 37"; } }
        attribute plannedStart : Real { @DataSource { sourceClass = SourceClass::NetworkShared; conopsRef = "UTM §2.7.1, p. 23; §3.2, p. 39"; } }
        attribute plannedEnd : Real;
    }
    item def OperationRecord {
        doc /* Archived record of a completed operation: actual times from the flight data sent to the
         * USS and archived as 4D position tracks (UTM ConOps Sec. 3.2, p. 39; Sec. 2.7.2, p. 31). */
        attribute intentId : String;
        attribute actualStart : Real { @DataSource { sourceClass = SourceClass::ProviderArchived; conopsRef = "UTM §3.2, p. 39; §2.7.2, p. 31"; } }
        attribute actualEnd : Real;
    }

    // ---------- Measurement parameters (open decisions, section 3.4 of the proposal) ----------

    attribute def MeasurementParameters {
        doc /* periodSeconds = aggregation period T; toleranceSeconds = tau for UTM-PRED-01;
         * shiftWindowSeconds = Delta, the negotiated shift still counted as accommodated in UTM-CAP-01. */
        attribute periodSeconds : Real;
        attribute toleranceSeconds : Real;
        attribute shiftWindowSeconds : Real;
    }

    // ---------- Performance Objectives (article 1, Table 3) ----------

    requirement def PO_Capacity {
        doc /* Accommodate valid operational demand within the available UTM operational environment. */
        @KPAApplicability { domain = Domain::UTM; applicability = Applicability::Applicable; }
        subject env : UTMEnvironment;
        frame concern kpa : Capacity;
    }
    requirement def PO_Predictability {
        doc /* Provide predictable operational outcomes with respect to planned timing. */
        @KPAApplicability { domain = Domain::UTM; applicability = Applicability::Applicable; }
        subject env : UTMEnvironment;
        frame concern kpa : Predictability;
    }

    // ---------- KPI candidates ----------

    calc def <'UTM-CAP-01'> DemandAccommodationRate :> AccommodationRate {
        doc /* accommodated = valid intents accepted by the USS (outcome Accepted, or Withdrawn after
         * acceptance) with plannedStart - requestedStart no greater than Delta; validDemand = valid
         * intents submitted in the period. PROPOSED. */
        @CandidateKPI {
            code = "UTM-CAP-01";
            domain = Domain::UTM;
            measurementConstruct = "Demand Accommodation";
            status = RetentionStatus::ProvisionallyRetained;
            maturity = Operationalization::ProposedInThisStudy;
        }
    }
    calc def <'UTM-PRED-01'> OperationTimePredictabilityRate :> TemporalConformanceRate {
        doc /* withinTolerance = completed operations with |actualStart - plannedStart| no greater than tau;
         * completed = operations with an archived record in the period. PROPOSED. */
        @CandidateKPI {
            code = "UTM-PRED-01";
            domain = Domain::UTM;
            measurementConstruct = "Temporal Predictability";
            status = RetentionStatus::ProvisionallyRetained;
            maturity = Operationalization::ProposedInThisStudy;
        }
    }

    // ---------- Supporting diagnostic metrics ----------

    calc def RejectionsByCause {
        doc /* Share of rejected intents per cause (conflict, constraint). Diagnostic of UTM-CAP-01. */
        @SupportingDiagnosticMetric;
        return metricValue : Real;
    }
    calc def IntentDensity {
        doc /* Valid intents per volume and period. Diagnostic of UTM-CAP-01. */
        @SupportingDiagnosticMetric;
        return metricValue : Real;
    }
    calc def StartDeviationP95 {
        doc /* 95th percentile of |actualStart - plannedStart|, in seconds. Diagnostic of UTM-PRED-01. */
        @SupportingDiagnosticMetric;
        return metricValue : Real;
    }

    // ---------- Traceability ----------

    dependency cap_measures_po from DemandAccommodationRate to PO_Capacity;
    dependency pred_measures_po from OperationTimePredictabilityRate to PO_Predictability;

    dependency cap_uses_validity from DemandAccommodationRate to OperationIntent::isValid { @KPIInputRole { role = InputRole::Supporting; } }
    dependency cap_uses_outcome from DemandAccommodationRate to OperationIntent::outcome { @KPIInputRole { role = InputRole::Principal; } }
    dependency cap_uses_requested from DemandAccommodationRate to OperationIntent::requestedStart { @KPIInputRole { role = InputRole::Supporting; } }
    dependency cap_uses_planned from DemandAccommodationRate to OperationIntent::plannedStart { @KPIInputRole { role = InputRole::Supporting; } }
    dependency pred_uses_planned from OperationTimePredictabilityRate to OperationIntent::plannedStart { @KPIInputRole { role = InputRole::Principal; } }
    dependency pred_uses_actual from OperationTimePredictabilityRate to OperationRecord::actualStart { @KPIInputRole { role = InputRole::Supporting; } }

    dependency rejections_explains from RejectionsByCause to DemandAccommodationRate;
    dependency density_explains from IntentDensity to DemandAccommodationRate;
    dependency deviation_explains from StartDeviationP95 to OperationTimePredictabilityRate;

    // ---------- KPI targets as verifiable requirements ----------

    requirement def CapacityTarget :> PO_Capacity {
        doc /* Refines PO_Capacity into a verifiable statement on UTM-CAP-01: the measured value shall be at
         * least the target. Target values are illustrative assumptions of the synthetic study. */
        attribute measured : Real;
        attribute target : Real;
        require constraint { measured >= target }
    }
    requirement def PredictabilityTarget :> PO_Predictability {
        doc /* Refines PO_Predictability into a verifiable statement on UTM-PRED-01. */
        attribute measured : Real;
        attribute target : Real;
        require constraint { measured >= target }
    }
    dependency capTarget_on_kpi from CapacityTarget to DemandAccommodationRate;
    dependency predTarget_on_kpi from PredictabilityTarget to OperationTimePredictabilityRate;

    // ---------- Synthetic study ----------

    part def SyntheticScenario {
        doc /* Configuration of one synthetic run (experiments E2 to E4 of the proposal). */
        attribute demandPerHour : Real;
        attribute ussCount : Natural;
        attribute constraintActive : Boolean;
        attribute messageFailureProbability : Real;
        attribute startDeviationSigmaSeconds : Real;
        attribute params : MeasurementParameters;
    }
    analysis def CapacityPredictabilityRun {
        doc /* One synthetic run: the generator produces intents and records for the scenario; the
         * counts below are taken from them and both KPIs are evaluated. */
        subject scenario : SyntheticScenario;
        objective {
            doc /* Observe UTM-CAP-01 and UTM-PRED-01 under the scenario configuration. */
        }
        in nAccommodated : Natural;
        in nValid : Natural;
        in nWithinTolerance : Natural;
        in nCompleted : Natural;
        out capResult : Real = DemandAccommodationRate(nAccommodated, nValid);
        out predResult : Real = OperationTimePredictabilityRate(nWithinTolerance, nCompleted, scenario.params.toleranceSeconds);
    }

    // ---------- Base case: analysis result bound to the targets ----------

    part baseCase : UTMEnvironment;
    analysis baseRun : CapacityPredictabilityRun;
    requirement baseCaseCapacity : CapacityTarget {
        attribute :>> measured = baseRun.capResult;
        attribute :>> target = 75.0;
    }
    requirement baseCasePredictability : PredictabilityTarget {
        attribute :>> measured = baseRun.predResult;
        attribute :>> target = 80.0;
    }
    satisfy baseCaseCapacity by baseCase;
    satisfy baseCasePredictability by baseCase;
}

## Visualizações

In [ ]:
%viz UTM_Selected::UTMEnvironment

In [ ]:
%viz UTM_Selected::PO_Capacity

In [ ]:
%viz UTM_Selected::DemandAccommodationRate

In [ ]:
%viz UTM_Selected::CapacityPredictabilityRun

In [ ]:
%viz UTM_Selected::CapacityTarget

In [ ]:
%viz UTM_Selected::baseCaseCapacity